In [1]:
import torch
import pandas as pd
import json
from tqdm import tqdm
from config import INDEX_UNTIL, LAM_VALUES, REAL_DIR, TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE
import gc
from fid import (
    build_prompt_map, build_clip_model, build_feat_model,
    compute_real_stats, compute_fid_score, compute_clip, sanity_check
)
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
INDEX_UNTIL = 3

replica_exchanges = [True, False]

PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/pt_test")
TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/tsr_test")

LAM_VALUES = [ 0.85,0.90,0.95,0.98]
LAM_FULL = LAM_VALUES + [1.0 + ( 1.0-l) for l in LAM_VALUES]

SWAP_ALGORITHM = {
	"n_replicas": 3,
	"p_ratio": "p",
	"even_indices": [0, 2, 4, 6, 8, 12],   # t ≈ 870, 763, 648, 536
	"odd_indices":  [1, 3, 5, 7, 9, 13],
	"debug": True,
}

# ── Load prompts once ─────────────────────────────────────────────────────────
df = pd.read_csv(PROMPTS_FILE, dtype={"original_idx": str})
prompts = df["text"].tolist()[:INDEX_UNTIL]
print(f"Loaded {len(prompts)} prompts")

Loaded 3 prompts


In [3]:
gc.collect()
torch.cuda.empty_cache()

In [4]:
from diffusers import StableDiffusion3Pipeline

# ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [ ]:
for replica_exchange in replica_exchanges:

	if replica_exchange:
		BASE_OUTPUT_DIR = PT_TSR_DIR
		LAM_ALL = LAM_VALUES
	else:
		BASE_OUTPUT_DIR = TSR_DIR
		LAM_ALL = LAM_FULL

	# ── Sweep ─────────────────────────────────────────────────────────────────────
	for tsr_lam in LAM_ALL:
		k_str = f"lam{tsr_lam:.3f}".replace(".", "p")   # e.g. "k0p950" — safe for filenames
		output_dir = BASE_OUTPUT_DIR / k_str
		checkpoint_file = output_dir / "completed.json"
		output_dir.mkdir(parents=True, exist_ok=True)

		if replica_exchange:
			k_str_flipped = f"lam{2.0-tsr_lam:.3f}".replace(".", "p")   # e.g. "k0p950" — safe for filenames
			output_dir_flipped = BASE_OUTPUT_DIR / k_str_flipped
			checkpoint_file_flipped = output_dir_flipped / "completed.json"
			output_dir_flipped.mkdir(parents=True, exist_ok=True)

		existing = list(output_dir.glob("*.png"))
		completed = set(range(len(existing)))
		if existing:
			print(f"\n[k={tsr_lam}] Resuming — {len(completed)}/{len(prompts)} done")
		else:
			completed = set()
			print(f"\n[k={tsr_lam}] Starting fresh")

		for idx, prompt in enumerate(tqdm(prompts, desc=f"k={tsr_lam}")):
			if idx in completed:
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED + idx)

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=TSR_SIGMA,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images

			out_path = output_dir / f"{df.iloc[idx]['original_idx']}.png"
			images[-1].save(out_path, icc_profile=None)

			if replica_exchange:
				out_path = output_dir_flipped / f"{df.iloc[idx]['original_idx']}.png"
				images[0].save(out_path, icc_profile=None)
			
			del images
			torch.cuda.empty_cache()

			completed.add(idx)
			if idx % 50 == 0:
				checkpoint_file.write_text(json.dumps(list(completed)))


		checkpoint_file.write_text(json.dumps(list(completed)))
		print(f"[k={tsr_lam}] Done — {len(completed)} images saved to {output_dir}")

	print("\n All k values complete.")


[k=0.85] Starting fresh


k=0.85:   0%|          | 0/3 [00:00<?, ?it/s]

We will be running with replica swaps with 3 replicas
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 1000.00
Time 1000.0 swap btwn source 1.00 and target 1.15 accept 1.000 std 0.992
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 988.27
Time 988.2713623046875 swap btwn source 0.85 and target 1.00 accept 0.953 std 0.979
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 975.98
Time 975.9791870117188 swap btwn source 1.00 and target 1.15 accept 1.000 std 0.967
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 963.08
Time 963.08203125 swap btwn source 0.85 and target 1.00 accept 0.994 std 0.955
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 949.53
Time 949.5339965820312 swap btwn source 1.00 and target 1.15 accept 0.999 std 0.943
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 935.28
Time 935.2844848632812 swap btwn source 0.85 and target 1.00 accept 0.988 std 0.931
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 920.28
Time 920.2777099609375 swap btwn sour

k=0.85:  33%|███▎      | 1/3 [00:39<01:18, 39.17s/it]

We will be running with replica swaps with 3 replicas
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 1000.00
Time 1000.0 swap btwn source 1.00 and target 1.15 accept 1.000 std 0.994
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 988.27
Time 988.2713623046875 swap btwn source 0.85 and target 1.00 accept 0.954 std 0.982
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 975.98
Time 975.9791870117188 swap btwn source 1.00 and target 1.15 accept 1.000 std 0.971
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 963.08
Time 963.08203125 swap btwn source 0.85 and target 1.00 accept 0.994 std 0.959
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 949.53
Time 949.5339965820312 swap btwn source 1.00 and target 1.15 accept 0.999 std 0.948
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 935.28
Time 935.2844848632812 swap btwn source 0.85 and target 1.00 accept 0.992 std 0.937
 We tsr by 1.15
 We tsr by 1.00
 We tsr by 0.85
t is 920.28
Time 920.2777099609375 swap btwn sour

...

We will be running with replica swaps with 3 replicas
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 1000.00
Time 1000.0 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.984
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 988.27
Time 988.2713623046875 swap btwn source 0.98 and target 1.00 accept 0.999 std 0.972
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 975.98
Time 975.9791870117188 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.959
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 963.08
Time 963.08203125 swap btwn source 0.98 and target 1.00 accept 1.000 std 0.947
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 949.53
Time 949.5339965820312 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.935
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 935.28
Time 935.2844848632812 swap btwn source 0.98 and target 1.00 accept 1.000 std 0.923
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 920.28
Time 920.2777099609375 swap btwn sour

k=0.98:  33%|███▎      | 1/3 [00:36<01:12, 36.48s/it]

We will be running with replica swaps with 3 replicas
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 1000.00
Time 1000.0 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.986
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 988.27
Time 988.2713623046875 swap btwn source 0.98 and target 1.00 accept 0.999 std 0.975
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 975.98
Time 975.9791870117188 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.962
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 963.08
Time 963.08203125 swap btwn source 0.98 and target 1.00 accept 1.000 std 0.951
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 949.53
Time 949.5339965820312 swap btwn source 1.00 and target 1.02 accept 1.000 std 0.939
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 935.28
Time 935.2844848632812 swap btwn source 0.98 and target 1.00 accept 1.000 std 0.927
 We tsr by 1.02
 We tsr by 1.00
 We tsr by 0.98
t is 920.28
Time 920.2777099609375 swap btwn sour

k=0.98:  67%|██████▋   | 2/3 [01:13<00:36, 36.56s/it]

We will be running with replica swaps with 3 replicas


In [ ]:
device = "cuda"

# ── Initialize models once ────────────────────────────────────────────────────
prompt_map                 = build_prompt_map()
clip_model, clip_processor = build_clip_model(device)
feat_model                 = build_feat_model(device)
mu_real, sigma_real        = compute_real_stats(feat_model, device, n=INDEX_UNTIL)

In [ ]:
# ── Compute ───────────────────────────────────────────────────────────────────
tsr_results = {alg: {} for alg in replica_exchanges}
TSR_DIRS = []

for alg in replica_exchanges:
	if alg == False:
		TSR_DIRS.append(TSR_DIR)
	elif alg == True:
		TSR_DIRS.append(PT_TSR_DIR)

for tsr_lam in LAM_FULL :
    lam_str = f"lam{tsr_lam:.3f}".replace(".", "p")
    print(f"\n── {lam_str} ──")

    for alg_idx, tsr_samples_dir in enumerate(TSR_DIRS):
        alg = replica_exchanges[alg_idx]
        samples_dir = (TSR_DIR if tsr_lam == 1.0 else tsr_samples_dir) / lam_str

        fid_val  = compute_fid_score(samples_dir, feat_model, mu_real, sigma_real, device, n=INDEX_UNTIL)
        clip_val = compute_clip(samples_dir, tsr_lam, prompt_map, clip_model, clip_processor, device, index_until=INDEX_UNTIL)
        tsr_results[alg][tsr_lam] = (fid_val, clip_val)
        print(f"[{alg}]  lam={tsr_lam:.3f}  FID={fid_val:.4f}  CLIP={clip_val:.4f}")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for alg in replica_exchanges:
    tsr_lam_vals    = sorted(tsr_results[alg].keys(), reverse=True)
    tsr_clip_vals = [tsr_results[alg][lam][1] for lam in tsr_lam_vals]
    tsr_fid_vals  = [tsr_results[alg][lam][0] for lam in tsr_lam_vals]

    ax.plot(tsr_clip_vals, tsr_fid_vals, marker="o", linewidth=2, label=f"{alg}, CFG=7.5, σ=3.0")
    for lam in tsr_lam_vals:
        f, c = tsr_results[alg][lam]
        ax.annotate(f"lam={lam}", (c, f), textcoords="offset points", xytext=(6, 0), fontsize=8, color="goldenrod")

ax.set_xlabel("CLIP", fontsize=12)
ax.set_ylabel("FID", fontsize=12)
ax.set_title("FID vs CLIP comparison", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fid_vs_clip.png", dpi=150)
plt.show()